## Pattern ISC - Post by Post Analysis

Given that each subject watched posts in a different order, we analyze the data post by post.
This allows us to see how brain responses vary for each specific post across subjects, across post type.

**Analysis plan inspiration from:**
> Chen, J., Leong, Y., Honey, C. et al. Shared memories reveal shared structure in neural activity across individuals. Nat Neurosci 20, 115–125 (2017). https://doi.org/10.1038/nn.4450

**Analysis Steps:**
1. Denoised BOLD data is loaded from the previous notebook.
2. Post timings are loaded from the metadata - e-prime event files for each subject, for each run.
3. Post BOLD timeseries is extracted - then averaged across TRs within each post to have one value per post per voxel per subject.

(1) per-subject/run event loading → (2) TR windows → (3) event-wise pattern extraction with a 5×5×5 searchlight

### Extract post-level average BOLD patterns

In [ ]:
from pathlib import Path
import importlib
from yy_fmri_kit.event_isc import roi
importlib.reload(roi)
from yy_fmri_kit.event_isc.roi import run_all_subjects
from yy_fmri_kit.io.find_files import build_denoised_runs_dict

In [ ]:
DERIV = Path("/path/to/data/derivatives/denoised")

Build a dict of denoised runs files per subject

In [ ]:
runs_dict = build_denoised_runs_dict(
    derivatives_dir=DERIV,
    space="MNI152NLin2009cAsym",
    desc_keywords="nltoolsClean")

print(runs_dict)

In [ ]:
OUT    = Path("/path/to/data/derivatives/postbypost/roi")
EVENTS = Path("/path/to/behavioral_analyses/behavioral_data_fmri/combined_events_with_bids.csv")
MASK   = Path("/path/to/data/brain_masks/pieman_a1_2mm.nii")

Extract post-level average BOLD patterns per subject, per scan (and potentially timeshift)

In [ ]:
run_all_subjects(
    runs_dict=runs_dict,
    events_path=EVENTS,
    out_dir=OUT,
    subject_col="bids_id",
    run_col="run",
    shift_tr=4,
    mask_path=mask,
    onset_col="onset_s",
    duration_col="duration_s",
    time_unit="seconds")

Quick QC for the output:

In [ ]:
import numpy as np

out_dir = Path("/path/to/data/derivatives/postbypost/roi")
npz_files = list(out_dir.rglob("*_desc-roi_patterns.npz"))
npz_files[:3]
f = np.load(npz_files[0], allow_pickle=True)
f.files

post_ids = f["post_ids"]
data = f["data"]
print(post_ids.shape)
print(data.shape)



QC plots of extracted data matrices are shown below. Each row is a post, each column is a voxel (feature). Color indicates BOLD response level (denoised but not z-scored)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(data, aspect="auto", cmap="RdBu_r")
plt.colorbar()
plt.xlabel("features")
plt.ylabel("posts")
plt.show()


### Compute post-level ISC patterns across subjects for ROI

In [ ]:
import importlib
from yy_fmri_kit.event_isc import load_aligned
importlib.reload(load_aligned)
from yy_fmri_kit.event_isc.load_aligned import build_data_list_for_run

from yy_fmri_kit.isc import compute
importlib.reload(compute)
from yy_fmri_kit.isc.compute import compute_isc

Build data matrix: posts x features (voxels) per subject and compute ISC per post type

In [ ]:
run_type = "AntiLeft"

subjects, post_ids, data_list = build_data_list_for_run(OUT, run_type)

isc_subjectwise, isc_mean = compute_isc(
    data_list,
    return_subjectwise=True,
    fisher_z=True,      # recommended
    standardize="zscore"  # this z-scores across posts (your “time” axis)
)

print(run_type, "data shape:", data_list[0].shape)
print("isc_subjectwise:", isc_subjectwise.shape)  # (N, F)
print("isc_mean:", isc_mean.shape)                # (F,)


In [ ]:
np.save(f"isc_{run_type}_subjectwise.npy", isc_subjectwise)  # (N,F)
np.save(f"isc_{run_type}_mean.npy", isc_mean)                # (F,)

Compare real ISC top shuffeled ISC by post_id

In [ ]:
x = data_list[0].copy()
perm = np.random.permutation(x.shape[0])
data_list_shuf = data_list.copy()
data_list_shuf[0] = x[perm]

isc_subj_shuf, _ = compute_isc(data_list_shuf, return_subjectwise=True, fisher_z=True)
sub_isc_shuf = np.nanmean(isc_subj_shuf, axis=1)

print("Real mean ISC:", sub_isc.mean())
print("Shuffled mean ISC:", sub_isc_shuf.mean())


In [ ]:
plt.scatter(isc_subj_shuf, sub_isc_shuf)

Full pipeline example:

In [ ]:
from nilearn import image, plotting
import numpy as np
isc_map_path = '/path/to/data/derivatives/searchlight/AntiRight_isc_map.nii.gz'
results = image.load_img(isc_map_path)

# Load your group mask to get the 3D dimensions and affine
# group_mask was created during your initial run
mask_img = image.load_img('/path/to/data/derivatives/searchlight/group_mask.nii.gz')
mask_data = mask_img.get_fdata()

# Initialize an empty 3D volume
isc_3d_data = np.zeros(mask_img.shape)

# 'results' is your ISC dictionary from analyzer.analyze_from_file()
# 'centers' are the voxel coordinates from extraction
for i, isc_val in enumerate(results['isc_mean']):
    x, y, z = results['centers'][i].astype(int)
    isc_3d_data[x, y, z] = isc_val

# Create the Nifti object
isc_nii = image.new_img_like(mask_img, isc_3d_data)


In [ ]:
from nilearn import image, plotting
import numpy as np
isc_map_path = '/path/to/data/derivatives/searchlight/AntiRight_isc_map.nii.gz'
isc_nii = image.load_img(isc_map_path)

plotting.plot_glass_brain(
    isc_nii, 
    display_mode='lyrz',  # Left, Right, Coronal, Axial
    colorbar=True, 
    threshold=0.05,       # Only show ISC > 0.05
    title='Searchlight ISC: ProLeft'
)

# Use a threshold like 0.04 to see the strongest signals from your pilot
plotting.plot_stat_map(
    isc_nii, 
    threshold=0.04, 
    display_mode='ortho', 
    colorbar=True,
    title='ISC: AntiRight (Pilot N=3)'
)

In [ ]:
# Assuming you have paths for both conditions
pro_nii = image.load_img('/path/to/data/derivatives/searchlight/ProLeft_isc_map.nii.gz')
anti_nii = image.load_img('/path/to/data/derivatives/searchlight/AntiRight_isc_map.nii.gz')

# Calculate the difference: Pro > Anti will be red, Anti > Pro will be blue
diff_img = image.math_img("img1 - img2", img1=pro_nii, img2=anti_nii)

plotting.plot_stat_map(
    diff_img, 
    threshold=0.02, 
    cmap='RdBu_r', 
    title='Polarization Difference: Pro vs Anti'
)

In [ ]:
import numpy as np
from nilearn import image, plotting

# 1. Load your results and group mask
# Replace these paths with your actual 12-subject results
isc_map_path = '/path/to/data/derivatives/searchlight/ProLeft_isc_map.nii.gz'
mask_path = '/path/to/data/derivatives/searchlight/group_mask.nii.gz'

isc_nii = image.load_img(isc_map_path)
mask_img = image.load_img(mask_path)
isc_data = isc_nii.get_fdata()

# 2. Identify the Top 5% threshold
# We only care about positive ISC values within the mask
valid_isc = isc_data[isc_data > 0]
top_threshold = np.percentile(valid_isc, 95)

print(f"Top 5% ISC Threshold: {top_threshold:.4f}")

# 3. Create a thresholded image
top_5_data = np.where(isc_data >= top_threshold, isc_data, 0)
top_5_nii = image.new_img_like(isc_nii, top_5_data)

# 4. Plot the "Hotspots"
plotting.plot_glass_brain(
    top_5_nii,
    display_mode='lyrz',
    colorbar=True,
    title='Top 5% Synchronized Searchlights (N=12)',
    plot_abs=False
)

# 5. Save the map for MRIcron
top_5_nii.to_filename('/path/to/data/derivatives/searchlight/Top5_Percent_ISC.nii.gz')